# Best directional action: reward and load change

This notebook uses the asymmetric one-step training scenario:

- Left gNB0 starts moderately loaded.
- Center gNB1 starts overloaded.
- Right gNB2 has the most available capacity.

We compare several directional upper actions and verify that requesting `center → right / eMBB` gives the best load balance and scaled causal reward.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_GAP_GNB_CONFIGS

GNB_NAMES = ['Left gNB0', 'Center gNB1', 'Right gNB2']
NEIGHBORS = {0: [1, 2], 1: [0, 2], 2: [0, 1]}

In [ ]:
def make_env(seed=2):
    return GlobalPPO3GNBEnv(
        seed=seed,
        scenario_mode='curriculum',
        training_scenarios='high_load_inner_asymmetric',
        scenario_selection='cycle',
        gnb_configs=CENTER_GAP_GNB_CONFIGS['medium_270m'],
        upper_window_seconds=1.0,
        local_steps_per_global=10,
        radio_substeps=20,
        terminal_reward_only=False,
        safe_admission_enabled=True,
    )

def run_action(label, left_bias=0.0, right_bias=0.0, seed=2):
    env = make_env(seed)
    try:
        observation, reset_info = env.reset(seed=seed)
        action = np.zeros(env.action_space.shape, dtype=np.float32)
        tensor = action.reshape(3, 2, 3)
        tensor[1, 0, 0] = left_bias   # center gNB1 -> left gNB0, eMBB
        tensor[1, 1, 0] = right_bias  # center gNB1 -> right gNB2, eMBB

        _, reward, terminated, truncated, info = env.step(action)
        events = list(env.base_env.get_handover_events())
        return {
            'label': label,
            'left_bias': left_bias,
            'right_bias': right_bias,
            'reward': float(reward),
            'scaled_load_reward': float(info['reward_load_improvement']),
            'saturation_reward': float(info['reward_saturation_improvement']),
            'raw_load_improvement': float(info['reward_load_improvement_raw']),
            'negative_bias_penalty': float(info['global_negative_bias_penalty']),
            'load_start': np.asarray(info['load_matrix_start'], dtype=float)[:, 0],
            'load_end': np.asarray(info['load_matrix_end'], dtype=float)[:, 0],
            'imbalance_start': float(info['load_imbalance_start']),
            'imbalance_end': float(info['load_imbalance_end']),
            'handovers': int(info['handover_count']),
            'routes': [(int(e['from_gnb']), int(e['to_gnb'])) for e in events],
            'direction_quota': dict(info['safe_admission']['direction_quota']),
            'direction_used': dict(info['safe_admission']['direction_used']),
            'terminated': bool(terminated),
            'truncated': bool(truncated),
            'observation_shape': observation.shape,
            'action_shape': action.shape,
        }
    finally:
        env.close()

## Compare candidate strategies

In [ ]:
cases = [
    run_action('Neutral'),
    run_action('Wrong: center → left', left_bias=-0.8),
    run_action('Both targets', left_bias=-0.8, right_bias=-0.8),
    run_action('Best: center → right', right_bias=-0.8),
]

comparison = pd.DataFrame([
    {
        'action': case['label'],
        'left bias': case['left_bias'],
        'right bias': case['right_bias'],
        'handovers': case['handovers'],
        'routes': str(case['routes']),
        'imbalance before': case['imbalance_start'],
        'imbalance after': case['imbalance_end'],
        'scaled load reward': case['scaled_load_reward'],
        'saturation reward': case['saturation_reward'],
        'action penalty': case['negative_bias_penalty'],
        'final reward': case['reward'],
    }
    for case in cases
])
comparison.style.format({
    'left bias': '{:.1f}', 'right bias': '{:.1f}',
    'imbalance before': '{:.3f}', 'imbalance after': '{:.3f}',
    'scaled load reward': '{:.3f}', 'saturation reward': '{:.3f}', 'action penalty': '{:.4f}',
    'final reward': '{:.3f}',
})

In [ ]:
best = max(cases, key=lambda case: case['reward'])
assert best['label'] == 'Best: center → right'
assert best['routes'] and set(best['routes']) == {(1, 2)}
assert best['reward'] > 0.0
assert best['imbalance_end'] < best['imbalance_start']
assert best['truncated'] is True
print('Best action:', best['label'])
print('Routes:', best['routes'])
print(f"Reward: {best['reward']:.3f}")

## Plot 1: reward for each action

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
labels = [case['label'] for case in cases]
rewards = [case['reward'] for case in cases]
colors = ['#adb5bd', '#e76f51', '#f4a261', '#2a9d8f']
bars = ax.bar(labels, rewards, color=colors)
ax.axhline(0.0, color='black', lw=0.8)
ax.set(title='Scaled causal reward by directional action', ylabel='reward')
ax.tick_params(axis='x', rotation=12)
ax.grid(axis='y', alpha=.25)
for bar, value in zip(bars, rewards):
    ax.text(bar.get_x() + bar.get_width()/2, value + 0.02, f'{value:.3f}', ha='center', weight='bold')
plt.tight_layout(); plt.show()

## Plot 2: per-gNB load before and after the best action

In [ ]:
x = np.arange(3)
width = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
before = ax.bar(x - width/2, best['load_start'], width, label='Before', color='#457b9d')
after = ax.bar(x + width/2, best['load_end'], width, label='After', color='#2a9d8f')
ax.set_xticks(x, GNB_NAMES)
ax.set_ylim(0, 1.0)
ax.set(ylabel='eMBB load', title='Best action: load change at each gNB')
ax.axhline(np.mean(best['load_start']), color='#e63946', ls='--', label='Balance target')
ax.legend(); ax.grid(axis='y', alpha=.25)
for bars in (before, after):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.02, f'{bar.get_height():.2f}', ha='center')
plt.tight_layout(); plt.show()

## Plot 3: load movement diagram

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
positions = np.array([-1.0, 0.0, 1.0])
ax.scatter(positions, [0, 0, 0], marker='^', s=240, color='#264653', zorder=3)
for idx, name in enumerate(GNB_NAMES):
    ax.text(positions[idx], .18, name, ha='center', weight='bold')
    ax.text(positions[idx], -.17, f"{best['load_start'][idx]:.2f} → {best['load_end'][idx]:.2f}", ha='center', fontsize=12)
ax.annotate(
    f"{best['handovers']} eMBB UEs",
    xy=(.92, .02), xytext=(.10, .02),
    arrowprops=dict(arrowstyle='->', lw=3, color='#2a9d8f'),
    color='#2a9d8f', ha='center', va='bottom', weight='bold',
)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-.35, .35); ax.axis('off')
ax.set_title('Directional action moves center load only toward the requested right target')
plt.show()

## Conclusion

For the allocated-utilization state `[0.51, 1.00, 0.12]`, the best action is:

```text
B[center gNB1, right gNB2, eMBB] = -0.8
B[center gNB1, left gNB0, eMBB]  = 0.0
```

Safe admission executes only the permitted rightward handovers. The dispersion term improves, while the saturation term prevents the agent from receiving the best reward by filling both target cells to 100%. In this deterministic check, center-to-right is preferred over center-to-both.